# New York City, 2024 → 2025: Did the Neighborhoods That Moved to Trump Move to Mamdani?

Part 6 found that the 2024 rightward shift in the bottom income quartile was carried by Hispanic and Black low-income counties; Parts 8–9 found the leavers were economically left, culturally conservative, price-focused, and did not see much economic difference between the candidates. Eleven months later, New York City nominated and elected a democratic socialist mayor on an explicitly economic platform (rent freeze, city-run groceries, free buses), against the best-funded establishment candidate in the race and with hostile coverage across the cable spectrum.

New York is the one place where the same precincts can be observed voting in both contests. The question is direct: **did the election districts that swung hardest to Trump in 2024 vote for Mamdani in 2025 — in the June Democratic primary and in the November general?** If they did, the 2024 leavers were reachable with an economic message; if Mamdani's coalition was the culturally liberal, college-educated core and the Trump-shifted neighborhoods stayed away or went to Cuomo, the Part 9 reading — that the cultural line held — extends to the case most often cited against it.

**Inputs:** `data/raw/nyc/nyc_ed_panel.csv` from `src/data_collection/nyc_pull.py` — ED-level 2024 presidential and 2025 mayoral general returns (BOE), 2025 primary first choices and Mamdani-vs-Cuomo head-to-head built from the cast vote record, and 2020 Census / ACS 2023 block-based demographics.

**Caveats up front.** This is ecological: an ED that moved to Trump and then to Mamdani need not contain the same people doing both. The primary electorate is registered Democrats only; the general electorate in an off-year mayoral race is smaller and older than a presidential one. ED lines changed slightly between the three contests (nyed_24d / 25b / 25d); EDs that do not match across files are dropped, and "combined" EDs (reported under another ED) are excluded.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import statsmodels.api as sm
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
RAW_DIR = Path('../data/raw'); TABLES_DIR = Path('../outputs/tables'); FIGURES_DIR = Path('../outputs/figures')
d = pd.read_csv(RAW_DIR / 'nyc' / 'nyc_ed_panel.csv', index_col='ElectDist')
print(d.shape); print(d.columns.tolist())
# 2020 presidential votes are crosswalked from pre-2022 ED lines to current EDs via 2020 census blocks (population-weighted); see nyc_pull.crosswalk_2020.
d['pres24_2p'] = d.pres24_harris + d.pres24_trump
d['harris_share'] = d.pres24_harris / d.pres24_2p
d['trump_share'] = d.pres24_trump / d.pres24_2p
d['gen25_2p'] = d.gen25_mamdani + d.gen25_cuomo
d['mamdani_gen_share'] = d.gen25_mamdani / d.gen25_total          # of all mayoral votes
d['mamdani_vs_cuomo_gen'] = d.gen25_mamdani / d.gen25_2p
d['sliwa_gen_share'] = d.gen25_sliwa / d.gen25_total
d['prim_h2h_total'] = d.prim25_h2h_mamdani + d.prim25_h2h_cuomo
d['mamdani_h2h_prim'] = d.prim25_h2h_mamdani / d.prim_h2h_total
d['mamdani_first_share'] = d.prim25_first_mamdani / d.prim25_ballots
d['cuomo_first_share'] = d.prim25_first_cuomo / d.prim25_ballots
d['turnout_ratio_25_24'] = d.gen25_total / d.pres24_total
d['pres20_2p'] = d.pres20_biden + d.pres20_trump
d['trump_share_20'] = d.pres20_trump / d.pres20_2p
d['swing_to_trump'] = d.trump_share - d.trump_share_20            # 2020 -> 2024, positive = moved right
ok = (d.pres24_2p >= 50) & (d.gen25_2p >= 50) & (d.pres20_2p >= 50) & (~d.combined_into.fillna(False)) & d.pop2020.gt(0)
e = d[ok].copy()
print('EDs usable (>=50 two-party votes in 2020, 2024, and 2025 generals, not combined, populated):', len(e))
print('Citywide 2020 (crosswalked, usable EDs): Trump %.1f%%; 2024: Trump %.1f%%; swing %+.1f pts' % (100*e.pres20_trump.sum()/e.pres20_2p.sum(), 100*e.pres24_trump.sum()/e.pres24_2p.sum(), 100*(e.pres24_trump.sum()/e.pres24_2p.sum() - e.pres20_trump.sum()/e.pres20_2p.sum())))
print('Citywide (usable EDs): Harris %.1f%%, Trump %.1f%% | Mamdani %.1f%% of mayoral vote, %.1f%% of Mamdani+Cuomo, Sliwa %.1f%% | primary Mamdani-Cuomo head-to-head %.1f%%' % (
    100*e.pres24_harris.sum()/e.pres24_2p.sum(), 100*e.pres24_trump.sum()/e.pres24_2p.sum(), 100*e.gen25_mamdani.sum()/e.gen25_total.sum(),
    100*e.gen25_mamdani.sum()/e.gen25_2p.sum(), 100*e.gen25_sliwa.sum()/e.gen25_total.sum(), 100*e.prim25_h2h_mamdani.sum()/e.prim_h2h_total.sum()))

(4326, 40)
['pres24_total', 'pres24_harris', 'pres24_trump', 'county', 'combined_into', 'gen25_total', 'gen25_mamdani', 'gen25_cuomo', 'gen25_sliwa', 'combined_into_g25', 'prim25_ballots', 'prim25_h2h_mamdani', 'prim25_h2h_cuomo', 'prim25_h2h_neither', 'prim25_first_mamdani', 'prim25_first_cuomo', 'prim25_first_lander', 'prim25_first_adams', 'prim25_first_stringer', 'prim25_first_myrie', 'prim25_first_ramos', 'prim25_first_blake', 'prim25_first_tilson', 'pres20_biden', 'pres20_trump', 'pres20_total', 'pop2020', 'hispanic_share', 'white_nh_share', 'black_nh_share', 'asian_nh_share', 'med_income', 'ba_share', 'poverty', 'renter', 'median_age', 'adult_share', 'age18_34', 'age35_64', 'age65p']
EDs usable (>=50 two-party votes in 2020, 2024, and 2025 generals, not combined, populated): 3870
Citywide 2020 (crosswalked, usable EDs): Trump 23.1%; 2024: Trump 31.0%; swing +7.9 pts
Citywide (usable EDs): Harris 69.0%, Trump 31.0% | Mamdani 51.0% of mayoral vote, 55.3% of Mamdani+Cuomo, Sliwa 7.0

## 1. The 2020→2024 swing and the 2025 maps

The 2024 shift is measured directly: the change in Trump's two-party share from 2020 (crosswalked to current ED lines through census blocks) to 2024. EDs are sorted into deciles of that swing and of the 2024 Trump level, and Mamdani's June and November 2025 performance is read off each.

In [2]:
def wcorr(x, y, w):
    m = x.notna() & y.notna() & w.notna()
    x, y, w = x[m], y[m], w[m]
    mx, my = np.average(x, weights=w), np.average(y, weights=w)
    return np.average((x-mx)*(y-my), weights=w) / np.sqrt(np.average((x-mx)**2, weights=w) * np.average((y-my)**2, weights=w))
print('ED-level correlations (weighted by 2024 two-party votes):')
print('%-24s %14s %16s' % ('', 'w/ 2024 Trump', 'w/ swing 20->24'))
for c in ['mamdani_gen_share', 'mamdani_vs_cuomo_gen', 'sliwa_gen_share', 'mamdani_h2h_prim', 'mamdani_first_share', 'cuomo_first_share', 'turnout_ratio_25_24']:
    print('%-24s %+14.3f %+16.3f' % (c, wcorr(e.trump_share, e[c], e.pres24_2p), wcorr(e.swing_to_trump, e[c], e.pres24_2p)))
print('corr(swing, 2024 Trump share) = %+.3f' % wcorr(e.swing_to_trump, e.trump_share, e.pres24_2p))
# deciles of the 2020->2024 swing, then of the 2024 level
e['swing_decile'] = pd.qcut(e.swing_to_trump, 10, labels=False) + 1
e['trump_decile'] = pd.qcut(e.trump_share, 10, labels=False) + 1
def decile_table(by):
  return e.groupby(by).apply(lambda s: pd.Series({
    'trump_share_20': s.pres20_trump.sum()/s.pres20_2p.sum(), 'trump_share_24': s.pres24_trump.sum()/s.pres24_2p.sum(), 'swing': s.pres24_trump.sum()/s.pres24_2p.sum() - s.pres20_trump.sum()/s.pres20_2p.sum(), 'mamdani_gen': s.gen25_mamdani.sum()/s.gen25_total.sum(), 'cuomo_gen': s.gen25_cuomo.sum()/s.gen25_total.sum(),
    'sliwa_gen': s.gen25_sliwa.sum()/s.gen25_total.sum(), 'mamdani_vs_cuomo_gen': s.gen25_mamdani.sum()/s.gen25_2p.sum(),
    'mamdani_h2h_prim': s.prim25_h2h_mamdani.sum()/s.prim_h2h_total.sum(), 'mamdani_first': s.prim25_first_mamdani.sum()/s.prim25_ballots.sum(),
    'prim_ballots_per_24_vote': s.prim25_ballots.sum()/s.pres24_total.sum(), 'gen25_per_24_vote': s.gen25_total.sum()/s.pres24_total.sum(),
    'hispanic': np.average(s.hispanic_share, weights=s.pop2020), 'black': np.average(s.black_nh_share, weights=s.pop2020), 'white': np.average(s.white_nh_share, weights=s.pop2020),
    'asian': np.average(s.asian_nh_share, weights=s.pop2020), 'med_income': np.average(s.med_income.fillna(s.med_income.median()), weights=s.pop2020), 'ba_share': np.average(s.ba_share.fillna(0), weights=s.pop2020), 'N': len(s)}))
decs = decile_table('swing_decile'); print('\nBy decile of the 2020->2024 swing to Trump (1 = moved least / left, 10 = moved most right):'); print(decs.round(3))
dec = decile_table('trump_decile'); print('\nBy decile of 2024 Trump share:'); print(dec.round(3))
decs.to_csv(TABLES_DIR / 'nyc_by_swing_decile.csv'); dec.to_csv(TABLES_DIR / 'nyc_by_trump_decile.csv')

ED-level correlations (weighted by 2024 two-party votes):
                          w/ 2024 Trump  w/ swing 20->24
mamdani_gen_share                -0.801           -0.099
mamdani_vs_cuomo_gen             -0.764           -0.067
sliwa_gen_share                  +0.737           +0.137
mamdani_h2h_prim                 -0.313           -0.051
mamdani_first_share              -0.182           +0.105
cuomo_first_share                +0.367           +0.124
turnout_ratio_25_24              +0.106           +0.125
corr(swing, 2024 Trump share) = +0.360

By decile of the 2020->2024 swing to Trump (1 = moved least / left, 10 = moved most right):
              trump_share_20  trump_share_24  swing  mamdani_gen  cuomo_gen  \
swing_decile                                                                  
1                      0.162           0.160 -0.003        0.638      0.323   
2                      0.176           0.202  0.026        0.563      0.383   
3                      0.190          

## 2. The Part 6 neighborhoods: Hispanic, Black, Asian, and low-income EDs

Part 6's 2024 movers were nonwhite low-income places. Here each ED is typed by its plurality group (2020 Census) and income tercile, and the three contests are compared within type.

In [3]:
e['plurality'] = e[['hispanic_share', 'white_nh_share', 'black_nh_share', 'asian_nh_share']].idxmax(axis=1).str.replace('_share', '').str.replace('_nh', '')
e['majority'] = np.where(e[['hispanic_share', 'white_nh_share', 'black_nh_share', 'asian_nh_share']].max(axis=1) >= 0.5, e.plurality, 'mixed')
e['income_tercile'] = pd.qcut(e.med_income, 3, labels=['low', 'mid', 'high'])
def agg(s):
    return pd.Series({'N': len(s), 'pop': s.pop2020.sum(), 'trump_20': s.pres20_trump.sum()/s.pres20_2p.sum(), 'trump_24': s.pres24_trump.sum()/s.pres24_2p.sum(), 'swing': s.pres24_trump.sum()/s.pres24_2p.sum() - s.pres20_trump.sum()/s.pres20_2p.sum(),
        'mamdani_gen': s.gen25_mamdani.sum()/s.gen25_total.sum(), 'cuomo_gen': s.gen25_cuomo.sum()/s.gen25_total.sum(), 'sliwa_gen': s.gen25_sliwa.sum()/s.gen25_total.sum(),
        'mamdani_vs_cuomo_gen': s.gen25_mamdani.sum()/s.gen25_2p.sum(), 'mamdani_h2h_prim': s.prim25_h2h_mamdani.sum()/s.prim_h2h_total.sum(),
        'mamdani_first': s.prim25_first_mamdani.sum()/s.prim25_ballots.sum(), 'cuomo_first': s.prim25_first_cuomo.sum()/s.prim25_ballots.sum(),
        'gen25_per_24_vote': s.gen25_total.sum()/s.pres24_total.sum(), 'prim_per_24_vote': s.prim25_ballots.sum()/s.pres24_total.sum()})
by_maj = e.groupby('majority').apply(agg); print('By majority group:'); print(by_maj.round(3))
by_inc = e.groupby('income_tercile').apply(agg); print('\nBy income tercile:'); print(by_inc.round(3))
cross = e.groupby(['majority', 'income_tercile']).apply(agg); print('\nMajority x income:'); print(cross.round(3))
by_maj.to_csv(TABLES_DIR / 'nyc_by_majority.csv'); cross.to_csv(TABLES_DIR / 'nyc_by_majority_x_income.csv')

By majority group:
               N        pop  trump_20  trump_24  swing  mamdani_gen  \
majority                                                              
asian      226.0   601437.0     0.321     0.469  0.148        0.465   
black      628.0  1356712.0     0.055     0.101  0.045        0.636   
hispanic   723.0  1805994.0     0.162     0.298  0.136        0.584   
mixed     1060.0  2342433.0     0.222     0.307  0.086        0.577   
white     1233.0  2561271.0     0.334     0.384  0.049        0.413   

          cuomo_gen  sliwa_gen  mamdani_vs_cuomo_gen  mamdani_h2h_prim  \
majority                                                                 
asian         0.424      0.104                 0.523             0.614   
black         0.329      0.026                 0.659             0.437   
hispanic      0.349      0.059                 0.626             0.553   
mixed         0.346      0.070                 0.625             0.646   
white         0.498      0.084         

## 3. Within-type: among EDs of the same demographic type, did the ones that swung to Trump go for Mamdani?

The between-type comparison mixes who lives there with how they moved. Within each majority group, regress Mamdani's 2025 measures on the 2020→2024 swing, holding the 2020 Trump level, income, education, and composition fixed. A positive coefficient on the swing means the EDs that moved right in 2024 were *more* Mamdani in 2025 than otherwise-similar EDs that did not; negative means the 2024 movers stayed away from him.

In [4]:
rows = []
for grp, s in list(e.groupby('majority')) + [('all', e)]:
    for dep in ['mamdani_vs_cuomo_gen', 'mamdani_h2h_prim', 'sliwa_gen_share', 'turnout_ratio_25_24']:
        w = s.pres24_2p if 'gen' in dep else s.prim_h2h_total
        sub = s.dropna(subset=[dep, 'swing_to_trump', 'trump_share_20', 'med_income', 'ba_share']); w = w.loc[sub.index]
        X = sm.add_constant(pd.DataFrame({'swing': sub.swing_to_trump, 'trump_20': sub.trump_share_20, 'log_income': np.log(sub.med_income), 'ba_share': sub.ba_share, 'hispanic': sub.hispanic_share, 'black': sub.black_nh_share, 'asian': sub.asian_nh_share}))
        r = sm.WLS(sub[dep], X, weights=w).fit(cov_type='HC1')
        rows.append({'group': grp, 'dep': dep, 'coef_swing': r.params['swing'], 'se_swing': r.bse['swing'], 'coef_trump20': r.params['trump_20'], 'se_trump20': r.bse['trump_20'], 'N': len(sub), 'R2': r.rsquared})
wt = pd.DataFrame(rows).set_index(['group', 'dep']).round(3)
print('Coefficients on the 2020->2024 swing and on the 2020 Trump level (0->1 changes), within group, controlling for log income, BA share, composition:'); print(wt)
print('Read: a swing coefficient of -0.5 means an ED that moved 10 points to Trump in 2024 was 5 points less Mamdani in 2025 than a similar ED that did not move.')
wt.to_csv(TABLES_DIR / 'nyc_within_group_models.csv')

Coefficients on the 2020->2024 swing and on the 2020 Trump level (0->1 changes), within group, controlling for log income, BA share, composition:
                               coef_swing  se_swing  coef_trump20  se_trump20  \
group    dep                                                                    
asian    mamdani_vs_cuomo_gen       0.034     0.103        -1.483       0.088   
         mamdani_h2h_prim           0.262     0.161        -1.419       0.098   
         sliwa_gen_share            0.032     0.041         0.320       0.037   
         turnout_ratio_25_24        0.620     0.180        -0.836       0.118   
black    mamdani_vs_cuomo_gen      -0.624     0.212        -1.650       0.123   
         mamdani_h2h_prim          -0.213     0.137        -2.328       0.201   
         sliwa_gen_share            0.022     0.021         0.319       0.043   
         turnout_ratio_25_24        7.730     2.268        -1.813       0.571   
hispanic mamdani_vs_cuomo_gen      -0.317   

## 3b. Against the rest of his own ballot

The recovery against Harris in the swing districts could be the party-wide rebound of 4 November 2025 (Democrats won New Jersey and Virginia by double digits the same day). The comptroller (Levine) and public advocate (Williams) races on the same ballot give a same-voters, same-day Democratic baseline: if they close on Harris across the swing gradient as fast as Mamdani does, the recovery is the ticket's, not his.

In [5]:
import sys
import statsmodels.formula.api as smf
sys.path.insert(0, str(RAW_DIR.parent.parent / 'src' / 'data_collection'))
from nyc_pull import read_boe_edlevel
METHOD = {'Public Counter', 'Manually Counted Emergency', 'Absentee / Military', 'Federal', 'Affidavit', 'Scattered'}
sb = pd.read_csv(RAW_DIR / 'nyc' / 'nyc_ed_panel.csv')
for k, f, nm in [('williams', '00000200000Citywide_Public_Advocate_Citywide_EDLevel.csv', 'Williams'),
                 ('levine', '00000300000Citywide_City_Comptroller_Citywide_EDLevel.csv', 'Levine')]:
    a = read_boe_edlevel(RAW_DIR / 'nyc' / '2025_general' / f); a = a[a['EDAD Status'] == 'IN-PLAY']
    v = a[~a['Unit Name'].isin(METHOD)]
    tot = v.groupby('ElectDist').Tally.sum(); dv = v[v['Unit Name'].str.contains(nm)].groupby('ElectDist').Tally.sum()
    sb[k] = sb.ElectDist.map(dv / tot)
sb['harris'] = sb.pres24_harris / sb.pres24_total
sb['mamdani'] = sb.gen25_mamdani / sb.gen25_total
sb['swing'] = 100 * (sb.pres24_trump / (sb.pres24_trump + sb.pres24_harris) - sb.pres20_trump / (sb.pres20_trump + sb.pres20_biden))
ok = sb[(sb.gen25_total >= 50) & (sb.pres24_total >= 50) & (sb.pres20_total > 50) & sb.levine.notna() & sb.williams.notna()
        & ~sb.combined_into.fillna(False).astype(bool) & ~sb.combined_into_g25.fillna(False).astype(bool)].copy()
print('EDs', len(ok))
for c in ['mamdani', 'levine', 'williams']: ok[c + '_vH'] = 100 * (ok[c] - ok.harris)
ok['mam_minus_lev'] = 100 * (ok.mamdani - ok.levine); ok['mam_minus_wil'] = 100 * (ok.mamdani - ok.williams)
ok['dec'] = pd.qcut(ok.swing, 10, labels=False) + 1
w = lambda g, c: np.average(g[c], weights=g.gen25_total)
T = ok.groupby('dec').apply(lambda g: pd.Series({'swing': w(g, 'swing'), 'harris': 100 * w(g, 'harris'), 'mamdani': 100 * w(g, 'mamdani'),
    'levine': 100 * w(g, 'levine'), 'williams': 100 * w(g, 'williams'), 'mam_vs_H': w(g, 'mamdani_vH'),
    'lev_vs_H': w(g, 'levine_vH'), 'wil_vs_H': w(g, 'williams_vH')}))
T['mam_minus_lev'] = T.mamdani - T.levine; T['mam_minus_wil'] = T.mamdani - T.williams
print(T.round(1).to_string())
T.to_csv(TABLES_DIR / 'nyc_2025_same_ballot_by_swing_decile.csv')
ctrl = ' + hispanic_share + black_nh_share + asian_nh_share + ba_share + np.log(med_income) + age18_34 + C(county)'
for y in ['mamdani_vH', 'levine_vH', 'williams_vH', 'mam_minus_lev', 'mam_minus_wil']:
    for lab, f in [('raw', f'{y} ~ swing'), ('controls', f'{y} ~ swing' + ctrl)]:
        m = smf.wls(f, data=ok, weights=ok.gen25_total).fit(cov_type='HC1')
        print(f'{y:14s} {lab:8s} swing {m.params.swing:+.3f} ({m.bse.swing:.3f}) p={m.pvalues.swing:.3g}')

EDs 3879
     swing  harris  mamdani  levine  williams  mam_vs_H  lev_vs_H  wil_vs_H  mam_minus_lev  mam_minus_wil
dec                                                                                                      
1     -0.2    83.7     63.8    85.1      82.1     -20.0       1.3      -1.6          -21.3          -18.3
2      2.4    79.3     56.4    80.7      77.4     -22.9       1.4      -1.9          -24.3          -21.0
3      3.5    76.4     52.3    78.2      75.1     -24.1       1.9      -1.3          -25.9          -22.8
4      4.8    69.8     48.0    72.9      69.8     -21.8       3.2       0.1          -25.0          -21.8
5      6.2    64.5     45.6    70.0      66.5     -18.9       5.5       2.1          -24.4          -20.9
6      7.8    62.9     44.4    69.8      65.9     -18.5       6.9       3.0          -25.5          -21.6
7      9.9    60.3     44.0    69.7      66.5     -16.3       9.5       6.2          -25.7          -22.5
8     12.2    60.3     48.0    71.9  

## 3c. Against other Democrats the same day: New Jersey and Virginia

Section 3b holds the electorate fixed but cannot separate two things: a national rebound on 4 November 2025, and Mamdani bringing out voters who then voted the whole ticket. New Jersey and Virginia voted the same day. New Jersey is the closest comparison — same media market, Hispanic cities that swung hard to Trump — and Virginia's ballot put a progressive (Hashmi, lieutenant governor) and a moderate (Spanberger, governor) in front of the same voters.

The tests and the criteria that would count against the claim "Mamdani did not win over existing swing voters any better than other Democrats" were written down before any New Jersey or Virginia data was pulled (appendix K.1c). The measure for every candidate is the WLS slope of (2025 share − Harris 2024 share) on the 2020→24 swing, weighted by 2025 votes. NYC is also collapsed to its 65 Assembly Districts so the comparison with 564 New Jersey municipalities and 133 Virginia localities is at a comparable scale. Data: `same_day_2025_pull.py`.

In [6]:
import sys
from pathlib import Path
import numpy as np, pandas as pd, statsmodels.formula.api as smf
sys.path.insert(0, str(RAW_DIR.parent.parent / 'src' / 'data_collection'))
from nyc_pull import read_boe_edlevel


def same_day_tests():
    """Kept in a function so its frames do not overwrite the notebook panel `d`."""

    RAW = RAW_DIR

    def slope(d, y, w, x='swing'):
        m = smf.wls(f'{y} ~ {x}', data=d, weights=d[w]).fit(cov_type='HC1')
        return m.params[x], m.bse[x], m.pvalues[x], int(m.nobs), np.average(d[y], weights=d[w])

    def swing(d, t24, h24, t20, b20):
        return 100 * (d[t24] / (d[t24] + d[h24]) - d[t20] / (d[t20] + d[b20]))

    # ---------------- NYC: counts by ED, then Assembly District ----------------
    METHOD = {'Public Counter', 'Manually Counted Emergency', 'Absentee / Military', 'Federal', 'Affidavit', 'Scattered'}
    p = pd.read_csv(RAW / 'nyc' / 'nyc_ed_panel.csv')
    for k, f, nm in [('williams', '00000200000Citywide_Public_Advocate_Citywide_EDLevel.csv', 'Williams'),
                     ('levine', '00000300000Citywide_City_Comptroller_Citywide_EDLevel.csv', 'Levine')]:
        a = read_boe_edlevel(RAW / 'nyc' / '2025_general' / f); a = a[a['EDAD Status'] == 'IN-PLAY']
        v = a[~a['Unit Name'].isin(METHOD)]
        p[k + '_tot'] = p.ElectDist.map(v.groupby('ElectDist').Tally.sum())
        p[k] = p.ElectDist.map(v[v['Unit Name'].str.contains(nm)].groupby('ElectDist').Tally.sum())
    ok = p[(p.gen25_total >= 50) & (p.pres24_total >= 50) & (p.pres20_total > 50) & p.levine.notna() & p.williams.notna()
           & ~p.combined_into.fillna(False).astype(bool) & ~p.combined_into_g25.fillna(False).astype(bool)].copy()
    ok['AD'] = ok.ElectDist // 1000
    cols = ['gen25_mamdani', 'gen25_total', 'levine', 'levine_tot', 'williams', 'williams_tot',
            'pres24_harris', 'pres24_trump', 'pres24_total', 'pres20_biden', 'pres20_trump']
    units = {'NYC ED': ok, 'NYC Assembly District': ok.groupby('AD')[cols].sum().reset_index()}

    rows = []
    for lab, d in units.items():
        d = d.copy()
        d['swing'] = swing(d, 'pres24_trump', 'pres24_harris', 'pres20_trump', 'pres20_biden')
        d['H'] = d.pres24_harris / d.pres24_total
        d['y_mamdani'] = 100 * (d.gen25_mamdani / d.gen25_total - d.H)
        d['y_levine'] = 100 * (d.levine / d.levine_tot - d.H)
        d['y_williams'] = 100 * (d.williams / d.williams_tot - d.H)
        d['retain'] = d.gen25_total / d.pres24_total
        for cand, y, cls in [('Mamdani', 'y_mamdani', 'democratic socialist'), ('Williams', 'y_williams', 'progressive'),
                             ('Levine', 'y_levine', 'mainstream')]:
            b, se, pv, n, lev = slope(d, y, 'gen25_total')
            rows.append({'place': lab, 'candidate': cand, 'class': cls, 'N': n, 'slope': b, 'se': se, 'p': pv, 'mean_vs_harris': lev})
        b, se, pv, n, lev = slope(d, 'retain', 'gen25_total')
        rows.append({'place': lab, 'candidate': 'turnout retention', 'class': '', 'N': n, 'slope': b, 'se': se, 'p': pv, 'mean_vs_harris': lev})
        if lab == 'NYC Assembly District':
            nyc_ad = d

    # ---------------- New Jersey municipalities ----------------
    nj = pd.read_csv(RAW / 'same_day_2025' / 'nj_municipalities.csv')
    nj['swing'] = swing(nj, 'pres24_rep', 'pres24_dem', 'pres20_rep', 'pres20_dem')
    nj['y_sherrill'] = 100 * (nj.gov25_dem / nj.gov25_total - nj.pres24_dem / nj.pres24_total)
    nj['retain'] = nj.gov25_total / nj.pres24_total
    b, se, pv, n, lev = slope(nj, 'y_sherrill', 'gov25_total')
    rows.append({'place': 'NJ municipality', 'candidate': 'Sherrill', 'class': 'mainstream', 'N': n, 'slope': b, 'se': se, 'p': pv, 'mean_vs_harris': lev})
    b, se, pv, n, lev = slope(nj, 'retain', 'gov25_total')
    rows.append({'place': 'NJ municipality', 'candidate': 'turnout retention', 'class': '', 'N': n, 'slope': b, 'se': se, 'p': pv, 'mean_vs_harris': lev})

    # ---------------- Virginia localities ----------------
    va = pd.read_csv(RAW / 'same_day_2025' / 'va_localities.csv')
    va['swing'] = swing(va, 'pres24_rep', 'pres24_dem', 'pres20_rep', 'pres20_dem')
    H = va.pres24_dem / va.pres24_total
    for k, cand, cls in [('gov25', 'Spanberger', 'mainstream'), ('ltgov25', 'Hashmi', 'progressive'), ('ag25', 'Jones', 'mainstream (scandal)')]:
        va['y_' + k] = 100 * (va[k + '_dem'] / va[k + '_total'] - H)
        b, se, pv, n, lev = slope(va, 'y_' + k, 'gov25_total')
        rows.append({'place': 'VA locality', 'candidate': cand, 'class': cls, 'N': n, 'slope': b, 'se': se, 'p': pv, 'mean_vs_harris': lev})
    va['retain'] = va.gov25_total / va.pres24_total
    b, se, pv, n, lev = slope(va, 'retain', 'gov25_total')
    rows.append({'place': 'VA locality', 'candidate': 'turnout retention', 'class': '', 'N': n, 'slope': b, 'se': se, 'p': pv, 'mean_vs_harris': lev})

    R = pd.DataFrame(rows)
    pd.set_option('display.width', 200)
    print(R.round(3).to_string(index=False))

    # ---------------- the pre-specified tests ----------------
    print('\nTEST 1  same ballot, VA: Hashmi - Spanberger on swing (counts against the claim if > 0, p < 0.05)')
    va['d_hs'] = va.y_ltgov25 - va.y_gov25
    va['d_js'] = va.y_ag25 - va.y_gov25
    for y, lab in [('d_hs', 'Hashmi - Spanberger'), ('d_js', 'Jones - Spanberger (reported only)')]:
        b, se, pv, n, lev = slope(va, y, 'gov25_total')
        print(f'   {lab:36s} {b:+.3f} (SE {se:.3f}, p={pv:.3g})   mean gap {lev:+.2f}')

    print('\nTEST 2  Mamdani (NYC Assembly Districts) vs the moderates (counts against if above both by > 2 combined SE)')
    m = R[(R.place == 'NYC Assembly District') & (R.candidate == 'Mamdani')].iloc[0]
    for cand in ['Sherrill', 'Spanberger']:
        o = R[R.candidate == cand].iloc[0]
        diff = m.slope - o.slope; cse = np.hypot(m.se, o.se)
        print(f'   Mamdani {m.slope:.3f} vs {cand} {o.slope:.3f}: difference {diff:+.3f}, combined SE {cse:.3f}, z = {diff / cse:+.2f}')

    print('\nTEST 3  turnout retention on swing (reported)')
    print(R[R.candidate == 'turnout retention'][['place', 'N', 'slope', 'se', 'mean_vs_harris']].round(4).to_string(index=False))

    print('\nContext: swing ranges (weighted mean, p90, max)')
    for lab, d, w in [('NYC AD', nyc_ad, 'gen25_total'), ('NJ muni', nj, 'gov25_total'), ('VA locality', va, 'gov25_total')]:
        print(f'   {lab:12s} mean {np.average(d.swing, weights=d[w]):5.1f}  p90 {d.swing.quantile(.9):5.1f}  max {d.swing.max():5.1f}  sd {d.swing.std():4.1f}')
    R.to_csv(TABLES_DIR / 'same_day_2025_swing_reversion.csv', index=False)



same_day_tests()


                place         candidate                class    N  slope    se     p  mean_vs_harris
               NYC ED           Mamdani democratic socialist 3879  0.870 0.027 0.000         -17.165
               NYC ED          Williams          progressive 3879  0.923 0.018 0.000           4.115
               NYC ED            Levine           mainstream 3879  0.825 0.040 0.000           6.861
               NYC ED turnout retention                      3879  0.035 0.005 0.000           0.904
NYC Assembly District           Mamdani democratic socialist   65  1.060 0.307 0.001         -17.993
NYC Assembly District          Williams          progressive   65  1.160 0.186 0.000           3.544
NYC Assembly District            Levine           mainstream   65  0.956 0.153 0.000           6.285
NYC Assembly District turnout retention                        65 -0.017 0.003 0.000           0.826
      NJ municipality          Sherrill           mainstream  564  1.137 0.084 0.000       

## 3d. Was the swing-back a backlash to immigration enforcement?

Part 8 found immigration the strongest pull on 2024's switchers, and the places that swung hardest to Trump swung back hardest in November 2025. If that recovery was a backlash to enforcement, it should be larger where enforcement's reach is greatest. ICE's own arrest records (Deportation Data Project, FOIA release through August 2026) cannot place arrests at town level — New Jersey logged about 6,500 between inauguration and the election, a third of at-large arrests are coded to the Newark field office and custodial arrests to county jails — and no 2025 CES exists. Exposure is therefore proxied by the non-citizen share of residents (ACS 2020–24, `acs_immigration_pull.py`), which marks the mixed-status communities enforcement reaches.

Written before the ACS pull: the backlash reading is supported if the non-citizen share predicts the 2025 Democratic gain over Harris net of the 2024 swing, Hispanic share and income (primary test, New Jersey's 558 matched municipalities; Virginia's 133 localities second), and the share should predict the 2024 swing *to* Trump as a check on the proxy. The 2021 governor's race — the same off-year office, before the enforcement campaign, with the same Republican nominee in New Jersey — is the benchmark for what an ordinary off-year looks like.

In [7]:
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf


def enforcement_test():
    """Kept in a function so its frames do not overwrite the notebook panel `d`."""
    R = RAW_DIR / 'same_day_2025'
    w = lambda g, c, wt: np.average(g[c], weights=g[wt])
    fit = lambda f, data, wt: smf.wls(f, data=data, weights=data[wt]).fit(cov_type='HC1')

    def prep(x):
        x['nonc'] = 100 * x.noncitizen_share; x['hisp'] = 100 * x.hispanic_share; x['linc'] = np.log(x.med_income)
        x['swing24'] = 100 * (x.pres24_rep / (x.pres24_rep + x.pres24_dem) - x.pres20_rep / (x.pres20_rep + x.pres20_dem))
        x['y25'] = 100 * (x.gov25_dem / x.gov25_total - x.pres24_dem / x.pres24_total)
        x['y21'] = 100 * (x.gov21_dem / x.gov21_total - x.pres20_dem / x.pres20_total)
        x['rep_keep25'] = 100 * x.gov25_rep / x.pres24_rep; x['rep_keep21'] = 100 * x.gov21_rep / x.pres20_rep
        x['dem_keep25'] = 100 * x.gov25_dem / x.pres24_dem; x['dem_keep21'] = 100 * x.gov21_dem / x.pres20_dem
        return x.dropna(subset=['nonc', 'hisp', 'linc'])

    nj = prep(pd.read_csv(R / 'nj_municipalities.csv').merge(pd.read_csv(R / 'acs_nj_matched.csv'), on=['county', 'key']))
    va = pd.read_csv(R / 'va_localities.csv'); a = pd.read_csv(R / 'acs_va_localities.csv')
    va['k'] = va.locality.str.lower(); a['k'] = a.locality.str.lower()
    va = prep(va.merge(a[['k', 'noncitizen_share', 'hispanic_share', 'med_income']], on='k'))
    va['county'] = 'va'
    print(f'Units: NJ {len(nj)} municipalities, VA {len(va)} localities')

    rows = []
    for st, x in [('NJ', nj), ('VA', va)]:
        chk = fit('swing24 ~ nonc + hisp + linc', x, 'pres24_total')
        pre = fit('y25 ~ swing24 + nonc + hisp + linc', x, 'gov25_total')
        rows.append(dict(state=st, check_swing24_on_nonc=chk.params.nonc, check_se=chk.bse.nonc,
                         test_nonc=pre.params.nonc, test_se=pre.bse.nonc, test_p=pre.pvalues.nonc))
        if st == 'NJ':
            fe = fit('y25 ~ swing24 + nonc + hisp + linc + C(county)', x, 'gov25_total')
            rows[-1].update(fe_nonc=fe.params.nonc, fe_se=fe.bse.nonc, fe_p=fe.pvalues.nonc)
        # the 2021 benchmark: same off-year office, before the enforcement campaign
        L = pd.concat([x.assign(yr=2021, y=x.y21, wt=x.gov21_total, sw=0.0),
                       x.assign(yr=2025, y=x.y25, wt=x.gov25_total, sw=x.swing24)])
        L['uid'] = L.county.astype(str) + L.get('key', L.get('k')).astype(str)
        L['is25'] = (L.yr == 2025).astype(float)
        for c in ['nonc', 'hisp']:
            L[c + '_x25'] = L[c] * L.is25
        for lab, f in [('raw', 'y ~ is25 + nonc_x25 + C(uid)'),
                       ('net of 2024 swing and Hispanic share', 'y ~ is25 + nonc_x25 + hisp_x25 + sw + C(uid)')]:
            m = smf.wls(f, data=L, weights=L.wt).fit(cov_type='cluster', cov_kwds={'groups': L.uid})
            rows[-1][f'did_{lab}'] = f'{m.params.nonc_x25:+.3f} ({m.bse.nonc_x25:.3f}), p={m.pvalues.nonc_x25:.2f}'
    T = pd.DataFrame(rows).set_index('state')
    print(T.round(3).T.to_string())
    T.to_csv(TABLES_DIR / 'enforcement_test.csv')

    for st, x in [('NJ', nj), ('VA', va)]:
        x = x.copy(); x['q'] = pd.qcut(x.nonc, 5, labels=[1, 2, 3, 4, 5])
        Q = x.groupby('q', observed=True).apply(lambda g: pd.Series({
            'noncitizen_pct': w(g, 'nonc', 'pres24_total'), 'swing24': w(g, 'swing24', 'pres24_total'),
            'dem_gain_2025': w(g, 'y25', 'gov25_total'), 'dem_gain_2021': w(g, 'y21', 'gov21_total'),
            'rep_kept_2021': w(g, 'rep_keep21', 'pres20_rep'), 'rep_kept_2025': w(g, 'rep_keep25', 'pres24_rep'),
            'dem_kept_2021': w(g, 'dem_keep21', 'pres20_dem'), 'dem_kept_2025': w(g, 'dem_keep25', 'pres24_dem')}))
        print(f'\n{st} by fifth of non-citizen share (percent; "kept" = governor votes as % of the prior presidential vote)')
        print(Q.round(1).to_string())
        Q.round(2).to_csv(TABLES_DIR / f'enforcement_offyear_{st.lower()}.csv')


enforcement_test()


Units: NJ 558 municipalities, VA 133 localities
state                                                         NJ                      VA
check_swing24_on_nonc                                      0.117                   0.192
check_se                                                   0.022                   0.102
test_nonc                                                  0.157                   0.215
test_se                                                    0.048                   0.092
test_p                                                     0.001                   0.019
fe_nonc                                                    0.072                     NaN
fe_se                                                      0.044                     NaN
fe_p                                                       0.101                     NaN
did_raw                                   +0.211 (0.061), p=0.00  +0.517 (0.088), p=0.00
did_net of 2024 swing and Hispanic share  -0.024 (0.103), p=0.

## 4. Turnout: who showed up in 2025

Mamdani's primary and general were both higher-turnout than recent mayoral races. The ratio of 2025 general votes to 2024 presidential votes, and of primary ballots to 2024 votes, by demographic type and by 2024 Trump decile, says whether the 2025 electorate was the 2024 electorate or a subset of it.

In [8]:
print('Ratio of 2025 mayoral general votes to 2024 presidential votes, by majority group:'); print(by_maj[['gen25_per_24_vote', 'prim_per_24_vote']].round(3))
print('\nBy 2024 Trump decile:'); print(dec[['gen25_per_24_vote', 'prim_ballots_per_24_vote']].round(3))
# Where did Mamdani's general-election votes come from vs Harris's 2024 votes? share of each candidate's citywide total by majority group
src = e.groupby('majority').apply(lambda s: pd.Series({'harris_24_votes': s.pres24_harris.sum()/e.pres24_harris.sum(), 'trump_24_votes': s.pres24_trump.sum()/e.pres24_trump.sum(),
    'mamdani_gen_votes': s.gen25_mamdani.sum()/e.gen25_mamdani.sum(), 'cuomo_gen_votes': s.gen25_cuomo.sum()/e.gen25_cuomo.sum(), 'sliwa_gen_votes': s.gen25_sliwa.sum()/e.gen25_sliwa.sum(),
    'mamdani_prim_first': s.prim25_first_mamdani.sum()/e.prim25_first_mamdani.sum(), 'cuomo_prim_first': s.prim25_first_cuomo.sum()/e.prim25_first_cuomo.sum()}))
print('\nComposition of each candidate\'s citywide vote by ED majority group:'); print(src.round(3))
src.to_csv(TABLES_DIR / 'nyc_vote_composition_by_majority.csv')

Ratio of 2025 mayoral general votes to 2024 presidential votes, by majority group:
          gen25_per_24_vote  prim_per_24_vote
majority                                     
asian                 0.785             0.302
black                 0.668             0.384
hispanic              0.694             0.346
mixed                 0.814             0.401
white                 0.917             0.434

By 2024 Trump decile:
              gen25_per_24_vote  prim_ballots_per_24_vote
trump_decile                                             
1                         0.824                     0.559
2                         0.772                     0.480
3                         0.816                     0.491
4                         0.816                     0.456
5                         0.803                     0.421
6                         0.759                     0.378
7                         0.761                     0.355
8                         0.781                   

## 4b. Mamdani against the 2024 Democratic baseline

The cleanest ED-level comparison is Mamdani's November share against Harris's: both are general-election shares of the whole electorate in the same precinct a year apart. Where Mamdani ran furthest behind Harris, Democrats defected (to Cuomo) or stayed home; where he matched her, the 2024 Democratic vote held and any 2024 Trump vote went to Cuomo or Sliwa.

In [9]:
e['mamdani_minus_harris'] = e.mamdani_gen_share - e.harris_share
e['cuomo_sliwa_minus_trump'] = (e.gen25_cuomo + e.gen25_sliwa) / e.gen25_total - e.trump_share
t = e.groupby('swing_decile').apply(lambda s: pd.Series({'harris_24': s.pres24_harris.sum()/s.pres24_2p.sum(), 'mamdani_25': s.gen25_mamdani.sum()/s.gen25_total.sum(),
    'mamdani_minus_harris': s.gen25_mamdani.sum()/s.gen25_total.sum() - s.pres24_harris.sum()/s.pres24_2p.sum(),
    'trump_24': s.pres24_trump.sum()/s.pres24_2p.sum(), 'cuomo_plus_sliwa_25': (s.gen25_cuomo.sum() + s.gen25_sliwa.sum())/s.gen25_total.sum()}))
print('By swing decile: Mamdani 2025 vs Harris 2024, Cuomo+Sliwa 2025 vs Trump 2024 (shares of each contest vote):'); print(t.round(3))
t.to_csv(TABLES_DIR / 'nyc_mamdani_vs_harris_by_swing_decile.csv')
tg = e.groupby('majority').apply(lambda s: pd.Series({'harris_24': s.pres24_harris.sum()/s.pres24_2p.sum(), 'mamdani_25': s.gen25_mamdani.sum()/s.gen25_total.sum(),
    'mamdani_minus_harris': s.gen25_mamdani.sum()/s.gen25_total.sum() - s.pres24_harris.sum()/s.pres24_2p.sum(), 'trump_24': s.pres24_trump.sum()/s.pres24_2p.sum(), 'cuomo_plus_sliwa_25': (s.gen25_cuomo.sum() + s.gen25_sliwa.sum())/s.gen25_total.sum()}))
print('\nBy majority group:'); print(tg.round(3))
rows = []
for grp, s in list(e.groupby('majority')) + [('all', e)]:
    sub = s.dropna(subset=['mamdani_minus_harris', 'swing_to_trump', 'trump_share_20', 'med_income', 'ba_share'])
    X = sm.add_constant(pd.DataFrame({'swing': sub.swing_to_trump, 'trump_20': sub.trump_share_20, 'log_income': np.log(sub.med_income), 'ba_share': sub.ba_share, 'hispanic': sub.hispanic_share, 'black': sub.black_nh_share, 'asian': sub.asian_nh_share}))
    r = sm.WLS(sub.mamdani_minus_harris, X, weights=sub.pres24_2p).fit(cov_type='HC1')
    rows.append({'group': grp, 'coef_swing': r.params['swing'], 'se_swing': r.bse['swing'], 'coef_trump20': r.params['trump_20'], 'coef_ba': r.params['ba_share'], 'coef_log_income': r.params['log_income'], 'N': len(sub), 'R2': r.rsquared})
mh = pd.DataFrame(rows).set_index('group').round(3); print('\n(Mamdani 2025 - Harris 2024) on the swing, within group, with controls:'); print(mh)
mh.to_csv(TABLES_DIR / 'nyc_mamdani_minus_harris_models.csv')

By swing decile: Mamdani 2025 vs Harris 2024, Cuomo+Sliwa 2025 vs Trump 2024 (shares of each contest vote):
              harris_24  mamdani_25  mamdani_minus_harris  trump_24  cuomo_plus_sliwa_25
swing_decile                                                                            
1                 0.840       0.638                -0.202     0.160                0.357
2                 0.798       0.563                -0.235     0.202                0.432
3                 0.774       0.526                -0.247     0.226                0.468
4                 0.708       0.479                -0.228     0.292                0.515
5                 0.651       0.454                -0.196     0.349                0.540
6                 0.643       0.444                -0.199     0.357                0.550
7                 0.616       0.440                -0.176     0.384                0.553
8                 0.616       0.480                -0.136     0.384                0.513
9 

## 4c. Age, and where the nominating electorate grew

The panel's demographics were income, education, race, poverty and tenure — no age, which left the most obvious explanation for the primary pattern untested. Section 4 established that the June electorate grew only in white (1.25×) and mixed (1.19×) districts and shrank in Black (0.97×) and Asian (0.92×) ones, and that primary growth *declines* across the 2024 swing gradient. That is the shape you would expect if the nomination were driven by a young, educated, recently-arrived electorate rather than by the outer-borough districts that had moved to Trump.

ACS 2023 five-year `B01001` (sex by age) and `B01002` (median age) are now aggregated to EDs on the same population-weighted block-to-block-group join as the other ACS variables. Shares are of the **18+ population** rather than of everyone, because the quantity of interest is the composition of a potential electorate: a district with many children would otherwise read as "old" purely through its under-18s.

In [10]:
e['prim_growth_25_21'] = e.prim25_ballots / e.prim21_ballots if 'prim21_ballots' in e.columns else np.nan
if 'prim21_ballots' not in e.columns:
    _h = pd.read_csv(RAW_DIR / 'nyc' / 'nyc_ed_history.csv', index_col='ElectDist')
    e = e.join(_h[['prim21_ballots']], how='left')
    e['prim_growth_25_21'] = e.prim25_ballots / e.prim21_ballots
age_ok = e.age18_34.notna() & e.prim25_ballots.gt(25) & e.prim21_ballots.gt(25)
a = e[age_ok].copy()
print(f'EDs with age data and >=25 primary ballots in both 2021 and 2025: {len(a)}')
print(f'citywide pop-weighted median age {np.average(a.median_age, weights=a.pop2020):.1f}; '
      f'18-34 share of adults {np.average(a.age18_34, weights=a.pop2020):.1%}')

a['age_quintile'] = pd.qcut(a.age18_34, 5, labels=[1, 2, 3, 4, 5])
q = a.groupby('age_quintile', observed=True).apply(lambda s: pd.Series({
    'N': len(s), 'pct_18_34': s.age18_34.mean()*100, 'median_age': s.median_age.mean(),
    'primary_growth_25_21': s.prim25_ballots.sum()/s.prim21_ballots.sum(),
    'Mamdani_prim_h2h': s.prim25_h2h_mamdani.sum()/(s.prim25_h2h_mamdani.sum()+s.prim25_h2h_cuomo.sum()),
    'Mamdani_Nov': s.gen25_mamdani.sum()/s.gen25_total.sum(),
    'swing_to_Trump_24': s.swing_to_trump.mean()*100,
    'BA_share': s.ba_share.mean()*100}))
print('\nBY QUINTILE OF 18-34 SHARE OF ADULTS (1 = oldest districts, 5 = youngest):')
print(q.round(3).to_string())
q.to_csv(TABLES_DIR / 'nyc_by_age_quintile.csv')

print('\nWLS, weighted by 2025 primary ballots, HC1 SEs; controls log income, BA share, race shares:')
rows = []
for dep, lab in [('mamdani_h2h_prim', 'Mamdani vs Cuomo, primary'),
                 ('prim_growth_25_21', 'primary turnout growth 2021->2025'),
                 ('mamdani_gen_share', 'Mamdani share, November')]:
    X = a[['age18_34', 'ba_share', 'white_nh_share', 'black_nh_share', 'hispanic_share']].copy()
    X['log_income'] = np.log(a.med_income)
    X = sm.add_constant(X.dropna())
    y = a.loc[X.index, dep]; w = a.loc[X.index, 'prim25_ballots']
    keep = y.notna() & np.isfinite(y)
    m = sm.WLS(y[keep], X[keep], weights=w[keep]).fit(cov_type='HC1')
    rows.append({'outcome': lab, 'N': int(m.nobs), 'age18_34_coef': m.params.age18_34,
                 'se': m.bse.age18_34, 'p': m.pvalues.age18_34,
                 'per_10pp_young': m.params.age18_34*0.10, 'ba_share_coef': m.params.ba_share, 'R2': m.rsquared})
    print(f"  {lab:34s} age18_34 {m.params.age18_34:+.3f} (SE {m.bse.age18_34:.3f}, p={m.pvalues.age18_34:.1e})"
          f"  -> +10pp young adults = {m.params.age18_34*10:+.1f} pts   R2={m.rsquared:.3f}")
pd.DataFrame(rows).to_csv(TABLES_DIR / 'nyc_age_models.csv', index=False)
print('\nAge and education both carry large independent coefficients; the swing to Trump in 2024 is')
print('non-monotonic in age (highest in the middle quintiles), so youth is not simply a proxy for it.')

EDs with age data and >=25 primary ballots in both 2021 and 2025: 3834
citywide pop-weighted median age 39.6; 18-34 share of adults 31.2%

BY QUINTILE OF 18-34 SHARE OF ADULTS (1 = oldest districts, 5 = youngest):
                  N  pct_18_34  median_age  primary_growth_25_21  Mamdani_prim_h2h  Mamdani_Nov  swing_to_Trump_24  BA_share
age_quintile                                                                                                                
1             767.0     16.784      48.799                 1.085             0.470        0.391              7.530    47.920
2             767.0     24.816      42.949                 1.063             0.516        0.452              8.792    37.566
3             766.0     29.790      39.577                 1.078             0.523        0.510              9.772    34.795
4             767.0     35.437      36.346                 1.157             0.590        0.570              9.711    37.561
5             767.0     47.633      

## 4d. Registration: did he grow the party, or turn out the party that was there?

Section 4c showed the nominating electorate grew where young adults live. Votes alone cannot say whether that was new people joining the party or the Democrats already enrolled voting at higher rates. NYSBOE enrollment by election district settles the second and constrains the first.

Two features of the calendar make this tractable. New York runs a **closed** primary, and the deadline to change party enrollment for the 24 June 2025 primary was **14 February 2025** — a change filed after that date was held and not entered until 1 July. The snapshot used as the baseline is **20 February 2025**, six days after that deadline, so it is very close to a census of the Democrats who were *eligible to nominate him*. Primary turnout as a share of that baseline is therefore a clean rate, not an approximation.

The registration question is the harder one. The next snapshot is **1 November 2025**, so the window brackets the campaign rather than bisecting it: someone newly registered (as opposed to re-enrolled) could still join and vote in the June primary up to roughly mid-June, while party-changers after 14 February could not vote until the general. Growth measured Feb→Nov therefore mixes registration that preceded the primary with registration that followed it, and no intermediate snapshot exists to separate them.

In [11]:
EN = RAW_DIR / 'nyc_enrollment' / 'nyc_enrollment_by_ed.csv'
if not EN.exists():
    print('enrollment file not on disk; run src/data_collection/nyc_enrollment_pull.py')
else:
    en = pd.read_csv(EN)
    FEB, NOV = '2025-02-20', '2025-11-01'
    dem = en[en.status == 'total'].pivot_table(index='ElectDist', columns='snapshot', values='DEM', aggfunc='sum')
    r = e.join(dem[[FEB, NOV]].rename(columns={FEB: 'dem_feb25', NOV: 'dem_nov25'}), how='inner')
    r['dem_growth'] = r.dem_nov25 / r.dem_feb25
    r['dem_added'] = r.dem_nov25 - r.dem_feb25
    r['prim_turnout_of_dems'] = r.prim25_ballots / r.dem_feb25
    r = r[(r.dem_feb25 >= 100) & r.mamdani_h2h_prim.notna() & r.age18_34.notna()].copy()
    print(f'EDs with enrollment and results: {len(r)}')
    cw = en[en.status == 'total'].groupby('snapshot').DEM.sum()
    print('\nCitywide Democratic enrollment (total, incl. inactive):')
    for k, v in cw.items(): print(f'   {k}: {int(v):,}')
    print(f'   Feb->Nov 2025: {cw[NOV]-cw[FEB]:+,.0f} ({100*(cw[NOV]/cw[FEB]-1):+.1f}%)')
    print('   Note the 2021 figure is on pre-2022 ED lines; only the citywide total is comparable.')

    r['mam_q'] = pd.qcut(r.mamdani_h2h_prim, 5, labels=[1, 2, 3, 4, 5])
    q = r.groupby('mam_q', observed=True).apply(lambda s: pd.Series({
        'N': len(s),
        'Mamdani_primary': s.prim25_h2h_mamdani.sum()/(s.prim25_h2h_mamdani.sum()+s.prim25_h2h_cuomo.sum()),
        'dem_feb25': s.dem_feb25.sum(), 'dem_nov25': s.dem_nov25.sum(),
        'enrollment_growth': s.dem_nov25.sum()/s.dem_feb25.sum(),
        'net_added': s.dem_added.sum(),
        'primary_turnout_of_eligible_dems': s.prim25_ballots.sum()/s.dem_feb25.sum(),
        'pct_18_34': s.age18_34.mean()*100}))
    print('\nBY QUINTILE OF MAMDANI PRIMARY SHARE (1 = weakest, 5 = strongest):')
    print(q.round(3).to_string())
    q.to_csv(TABLES_DIR / 'nyc_enrollment_by_mamdani_quintile.csv')

    print('\nWLS, weighted by Feb-2025 Democratic enrollment, HC1:')
    rows = []
    for ctrl, lab in [([], 'enrollment growth ~ Mamdani share'),
                      (['age18_34', 'ba_share', 'white_nh_share', 'black_nh_share', 'hispanic_share'],
                       '  + age, education, race, income')]:
        X = r[['mamdani_h2h_prim'] + ctrl].copy()
        if ctrl: X['log_income'] = np.log(r.med_income)
        X = sm.add_constant(X.dropna())
        y = r.loc[X.index, 'dem_growth']; w = r.loc[X.index, 'dem_feb25']
        k = y.notna() & np.isfinite(y)
        m = sm.WLS(y[k], X[k], weights=w[k]).fit(cov_type='HC1')
        rows.append({'spec': lab, 'coef': m.params.mamdani_h2h_prim, 'se': m.bse.mamdani_h2h_prim,
                     'p': m.pvalues.mamdani_h2h_prim, 'R2': m.rsquared, 'N': int(m.nobs)})
        extra = f"   age18_34 {m.params.age18_34:+.4f}" if ctrl else ''
        print(f"  {lab:38s} {m.params.mamdani_h2h_prim:+.4f} (SE {m.bse.mamdani_h2h_prim:.4f}, "
              f"p={m.pvalues.mamdani_h2h_prim:.1e})  R2={m.rsquared:.3f}{extra}")
    pd.DataFrame(rows).to_csv(TABLES_DIR / 'nyc_enrollment_models.csv', index=False)
    print('\nBoth mechanisms operated. The turnout gradient is cleanly measured against a baseline')
    print('six days after the enrollment deadline; the registration gradient is directionally clear')
    print('but cannot be split into pre- and post-primary halves with the snapshots that exist.')

EDs with enrollment and results: 3848

Citywide Democratic enrollment (total, incl. inactive):
   2021-11-01: 3,780,378
   2025-02-20: 3,343,648
   2025-11-01: 3,502,856
   2026-02-20: 3,518,803
   Feb->Nov 2025: +159,208 (+4.8%)
   Note the 2021 figure is on pre-2022 ED lines; only the citywide total is comparable.

BY QUINTILE OF MAMDANI PRIMARY SHARE (1 = weakest, 5 = strongest):
           N  Mamdani_primary  dem_feb25  dem_nov25  enrollment_growth  net_added  primary_turnout_of_eligible_dems  pct_18_34
mam_q                                                                                                                         
1      770.0            0.247   674252.0   694952.0              1.031    20700.0                             0.257     27.610
2      769.0            0.399   634977.0   650018.0              1.024    15041.0                             0.257     27.871
3      770.0            0.512   612393.0   634801.0              1.037    22408.0                         

## 4e. Replication: the June 2026 congressional primaries

The limitation this Part has carried from the start is that New York is one candidate in one year. Eight months after Mamdani's general election, five Democratic House primaries inside the city put a candidate running to the left against an establishment one, over the same election districts: Valdez (DSA) for the open NY-7; Lander against the incumbent Goldman in NY-10; Avila Chevalier (DSA) against Espaillat in NY-13; Park against Meng in NY-6; Blake against Torres in NY-15. The left candidate is classified by the candidates' own positioning, not by who won.

Three questions, each a direct test of something Part 10 claimed:

1. **Is the coalition that nominated Mamdani a durable geography, or was it his?** Within each race, how well does his 2025 primary share predict the 2026 left vote, district by district?
2. **Did the mobilization persist?** In 2025 his strongest districts turned out far more of their enrolled Democrats than his weakest. The February 2026 enrolment snapshot sits six days after that cycle's 14 February party-change deadline, so it gives the same clean denominator for 2026.
3. **Did the swing districts join?** Part 10 found his primary strength was *not* in the districts that moved to Trump. Does the 2024 swing predict the 2026 left vote?

Data: `nyc_2026_primary_pull.py`. District lines were edited between November 2025 and February 2026 — 6.9% of districts changed their registered total by more than 25%, in both directions — so those districts are excluded from the main estimates, where the same number denotes different streets, and restored in a robustness check.

In [12]:
R26 = RAW_DIR / 'nyc' / 'nyc_2026_congressional_primary_by_ed.csv'
if not R26.exists():
    print('run src/data_collection/nyc_2026_primary_pull.py first')
else:
    import statsmodels.formula.api as smf
    r26 = pd.read_csv(R26)
    en26 = pd.read_csv(RAW_DIR / 'nyc_enrollment' / 'nyc_enrollment_by_ed.csv')
    demv = en26[en26.status == 'total'].pivot_table(index='ElectDist', columns='snapshot', values='DEM', aggfunc='sum')
    z = r26.merge(d.reset_index(), on='ElectDist', how='inner')
    z['dem_feb25'] = z.ElectDist.map(demv['2025-02-20'])
    z['mam_h2h'] = z.prim25_h2h_mamdani / (z.prim25_h2h_mamdani + z.prim25_h2h_cuomo)
    z['to25'] = z.prim25_ballots / z.dem_feb25
    z['to26'] = z.ballots26 / z.dem_enrolled_feb26
    base_ok = ((z.votes26 >= 25) & z.mam_h2h.notna() & (z.prim25_ballots >= 25) & z.swing_to_trump.notna()
               & (~z.combined_into.fillna(False)) & z.to25.between(0, 1.2) & z.to26.between(0, 1.2))
    ALL = z[base_ok].copy(); M = ALL[~ALL.lines_changed].copy()
    print(f'EDs: {len(ALL)} usable; {len(M)} after excluding the {int(ALL.lines_changed.sum())} with edited lines')

    def wc(x, y, w):
        mx, my = np.average(x, weights=w), np.average(y, weights=w)
        return np.average((x-mx)*(y-my), weights=w) / np.sqrt(np.average((x-mx)**2, weights=w) * np.average((y-my)**2, weights=w))

    print('\n1. SAME GEOGRAPHY? Within-race correlation, 2026 left head-to-head vs Mamdani 2025 head-to-head')
    rows = []
    for cd, g in M.groupby('cd'):
        ga = ALL[ALL.cd == cd]
        L = g.left_votes.sum() / (g.left_votes.sum() + g.est_votes.sum())
        Mm = g.prim25_h2h_mamdani.sum() / (g.prim25_h2h_mamdani.sum() + g.prim25_h2h_cuomo.sum())
        rows.append({'cd': cd, 'left_candidate': g.left_candidate.iloc[0], 'tier': g.tier.iloc[0],
                     'challenges_incumbent': g.challenges_incumbent.iloc[0], 'N': len(g),
                     'r_mamdani': wc(g.left_h2h, g.mam_h2h, g.votes26),
                     'r_mamdani_all_EDs': wc(ga.left_h2h, ga.mam_h2h, ga.votes26),
                     'r_swing': wc(g.left_h2h, g.swing_to_trump, g.votes26),
                     'mamdani_2025': Mm, 'left_2026': L, 'change_pts': 100 * (L - Mm)})
    R1 = pd.DataFrame(rows); print(R1.round(3).to_string(index=False))
    R1.to_csv(TABLES_DIR / 'nyc_2026_replication_by_race.csv', index=False)

    print('\n2. POOLED, race fixed effects, weighted by 2026 votes, HC1')
    specs = [('Mamdani 2025 only', 'left_h2h ~ mam_h2h + C(cd)'),
             ('demographics only', 'left_h2h ~ age18_34 + ba_share + swing_to_trump + hispanic_share + black_nh_share + asian_nh_share + np.log(med_income) + C(cd)'),
             ('both', 'left_h2h ~ mam_h2h + age18_34 + ba_share + swing_to_trump + hispanic_share + black_nh_share + asian_nh_share + np.log(med_income) + C(cd)')]
    out = []
    for lab, f in specs:
        for sample, data in [('lines stable', M), ('all EDs', ALL)]:
            m = smf.wls(f, data=data, weights=data.votes26).fit(cov_type='HC1')
            row = {'model': lab, 'sample': sample, 'N': int(m.nobs), 'R2': m.rsquared}
            for k in ['mam_h2h', 'age18_34', 'ba_share', 'swing_to_trump']:
                if k in m.params: row[k] = m.params[k]; row[k + '_se'] = m.bse[k]
            out.append(row)
    R2 = pd.DataFrame(out); print(R2.round(3).to_string(index=False))
    R2.to_csv(TABLES_DIR / 'nyc_2026_replication_models.csv', index=False)

    print('\n3. DID THE MOBILIZATION PERSIST? turnout of enrolled Democrats, by quintile of Mamdani 2025 share within race')
    M['mam_q'] = M.groupby('cd').mam_h2h.transform(lambda s: pd.qcut(s, 5, labels=[1, 2, 3, 4, 5]))
    Q = M.groupby('mam_q', observed=True).apply(lambda s: pd.Series({
        'N': len(s),
        'Mamdani_2025': s.prim25_h2h_mamdani.sum() / (s.prim25_h2h_mamdani.sum() + s.prim25_h2h_cuomo.sum()),
        'left_2026': s.left_votes.sum() / (s.left_votes.sum() + s.est_votes.sum()),
        'turnout_2025': s.prim25_ballots.sum() / s.dem_feb25.sum(),
        'turnout_2026': s.ballots26.sum() / s.dem_enrolled_feb26.sum()}))
    Q['kept_26_of_25'] = Q.turnout_2026 / Q.turnout_2025
    print(Q.round(3).to_string())
    g25 = Q.turnout_2025.iloc[-1] / Q.turnout_2025.iloc[0]; g26 = Q.turnout_2026.iloc[-1] / Q.turnout_2026.iloc[0]
    print(f'   strongest vs weakest quintile turnout: 2025 {g25:.2f}x, 2026 {g26:.2f}x')
    Q.to_csv(TABLES_DIR / 'nyc_2026_replication_turnout.csv')

    print('\n4. THE SWING DISTRICTS')
    for lab, f in [('swing alone', 'left_h2h ~ swing_to_trump + C(cd)'),
                   ('swing, net of Mamdani 2025', 'left_h2h ~ swing_to_trump + mam_h2h + C(cd)')]:
        m = smf.wls(f, data=M, weights=M.votes26).fit(cov_type='HC1')
        print(f'   {lab:28s} {m.params.swing_to_trump:+.3f} (SE {m.bse.swing_to_trump:.3f}, p={m.pvalues.swing_to_trump:.4f})')

EDs: 1639 usable; 1520 after excluding the 119 with edited lines

1. SAME GEOGRAPHY? Within-race correlation, 2026 left head-to-head vs Mamdani 2025 head-to-head
 cd            left_candidate        tier  challenges_incumbent   N  r_mamdani  r_mamdani_all_EDs  r_swing  mamdani_2025  left_2026  change_pts
  6                Chuck Park progressive                  True 261      0.872              0.860    0.089         0.508      0.428      -7.957
  7             Claire Valdez         DSA                 False 308      0.917              0.918   -0.025         0.762      0.624     -13.817
 10               Brad Lander progressive                  True 345      0.958              0.950   -0.207         0.687      0.670      -1.654
 13 Darializa Avila Chevalier         DSA                  True 341      0.899              0.901   -0.467         0.602      0.522      -7.907
 15             Michael Blake progressive                  True 265      0.695              0.682   -0.026         0.3

## 4f. Beyond the newspapers: YouTube and the creator ecosystem

The media sections above measure newspapers and a news index; 81% of CES respondents report getting news from social media, and the *Times* credited Mamdani with working the podcast circuit that helped Trump with young men. `youtube_creator_pull.py` collects, for 25 candidate-races in 2025–26 (the NYC mayoral primary and general, the five June 2026 NYC primaries, and the Michigan and Illinois Senate, CO-1, MI-13 and MO-1 primaries), the YouTube videos naming the candidate in the 90 days before the election, with views, split by channel type: legacy news, legacy entertainment, creators (online-first podcasts, streams, commentary, independent outlets) and political (campaigns, parties, PACs and paid promotion). The channel rule set in advance failed validation before any comparison was run and was replaced by hand-coding of the 104 channels carrying 95% of views (see the script).

Tests fixed before the pull: **H1** the left candidate draws a larger creator share of news-plus-creator views than the establishment opponent; **H2** across the five NYC June 2026 races, the left candidate's within-race gain in young districts is larger where the left candidate's creator reach exceeds the opponent's (five races: a rank comparison only).

In [13]:
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf


def creator_tests():
    """Kept in a function so its frames do not overwrite the notebook panel `d`."""
    v = pd.read_csv(RAW_DIR / 'youtube' / 'videos.csv')
    V = v.pivot_table(index=['race', 'candidate', 'side'], columns='kind', values='views', aggfunc='sum', fill_value=0)
    for k in ['creator', 'news', 'entertainment', 'political']:
        if k not in V: V[k] = 0
    V['creator_share'] = V.creator / (V.creator + V.news)
    V['total'] = V[['creator', 'news', 'entertainment', 'political']].sum(axis=1)
    V = V.reset_index()
    print('Views by channel type, millions (videos naming the candidate, 90 days before the election)')
    print((V.set_index(['race', 'candidate', 'side'])[['creator', 'news', 'entertainment', 'political', 'total']] / 1e6)
          .round(2).join(V.set_index(['race', 'candidate', 'side']).creator_share.round(2)).to_string())
    V.to_csv(TABLES_DIR / 'youtube_reach_by_candidate.csv', index=False)

    # H1: left vs establishment creator share, paired by race
    P = V[V.side.isin(['left', 'establishment'])].pivot_table(index='race', columns='side',
                                                               values=['creator_share', 'creator'])
    P.columns = [f'{a}_{b}' for a, b in P.columns]
    P['left_higher_share'] = P.creator_share_left > P.creator_share_establishment
    P['creator_views_ratio'] = P.creator_left / P.creator_establishment.replace(0, np.nan)
    print('\nH1. Creator share of news+creator views, left vs establishment')
    print(P[['creator_share_left', 'creator_share_establishment', 'left_higher_share', 'creator_views_ratio']].round(2).to_string())
    print(f"    left candidate has the higher creator share in {int(P.left_higher_share.sum())} of {len(P)} races; "
          f"higher creator views in {int((P.creator_views_ratio > 1).sum())} of {len(P)}")
    P.to_csv(TABLES_DIR / 'youtube_h1_paired.csv')

    # H2: NYC June 2026 races -- youth gradient of the left vote vs creator-reach advantage
    r26 = pd.read_csv(RAW_DIR / 'nyc' / 'nyc_2026_congressional_primary_by_ed.csv')
    z = r26.merge(d.reset_index()[['ElectDist', 'age18_34']], on='ElectDist')
    z = z[(z.votes26 >= 25) & ~z.lines_changed & z.age18_34.notna()]
    rows = []
    for cd, g in z.groupby('cd'):
        m = smf.wls('left_h2h ~ age18_34', data=g, weights=g.votes26).fit(cov_type='HC1')
        race = f'2026 NY-{cd}'
        rows.append(dict(race=race, left=g.left_candidate.iloc[0], youth_slope=m.params.age18_34 * 10,
                         youth_se=m.bse.age18_34 * 10,
                         creator_adv=np.log(P.loc[race, 'creator_views_ratio']) if race in P.index else np.nan))
    H = pd.DataFrame(rows)
    print('\nH2. NYC June 2026: left share per +10 points of 18-34 share (within race) vs log creator-views ratio (left/est)')
    print(H.round(3).to_string(index=False))
    print(f"    Spearman rank correlation, 5 races: {H[['youth_slope', 'creator_adv']].corr(method='spearman').iloc[0, 1]:+.2f}")
    H.to_csv(TABLES_DIR / 'youtube_h2_nyc.csv', index=False)


creator_tests()


Views by channel type, millions (videos naming the candidate, 90 days before the election)
                                                                creator   news  entertainment  political  total  creator_share
race                   candidate                 side                                                                         
2025 NYC mayor general Andrew Cuomo              establishment    11.29  13.77           8.15       2.04  35.25           0.45
                       Curtis Sliwa              other             4.05  14.57           8.15       2.45  29.22           0.22
                       Zohran Mamdani            left             44.59  25.92          20.71       2.55  93.76           0.63
2025 NYC mayor primary Andrew Cuomo              establishment     7.92   4.23           0.52       0.99  13.66           0.65
                       Zohran Mamdani            left             18.40  11.07           1.04       2.70  33.21           0.62
2026 CO-1           

## 5. Figure

In [14]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, dep, lab in [(axes[0], 'mamdani_vs_cuomo_gen', 'Mamdani share of Mamdani+Cuomo, Nov 2025'), (axes[1], 'mamdani_h2h_prim', 'Mamdani vs Cuomo head-to-head, June 2025 primary')]:
    for grp, col in [('hispanic', 'tab:orange'), ('black', 'tab:purple'), ('white', 'tab:blue'), ('asian', 'tab:green'), ('mixed', 'grey')]:
        s = e[e.majority == grp]
        ax.scatter(s.swing_to_trump, s[dep], s=np.sqrt(s.pres24_2p)/2, alpha=0.35, c=col, label=f'{grp} majority (N={len(s)})', edgecolors='none')
    ax.set_xlabel('Swing to Trump, 2020 -> 2024 (two-party share points)'); ax.set_ylabel(lab); ax.set_xlim(-0.15, 0.45); ax.set_ylim(0, 1); ax.grid(alpha=0.3); ax.axvline(0, color='k', lw=0.5)
axes[0].legend(fontsize=8, loc='upper right')
fig.suptitle('NYC election districts: 2020->2024 swing to Trump vs 2025 Mamdani vote')
plt.tight_layout(); plt.savefig(FIGURES_DIR / 'nyc_trump24_vs_mamdani25.png', dpi=150); print('saved figure')

saved figure


## Limitations

- **Ecological.** ED-level co-movement does not identify individual voters. A Hispanic ED that went 40% Trump in 2024 and 60% Mamdani in 2025 could be the same voters or different subsets of the same neighborhood.
- **The 2020 baseline is crosswalked** from the pre-2022 ED lines through 2020 census blocks, assuming votes are distributed within a 2020 ED in proportion to block population; 98% of 2020 votes are carried over. Swings in small or oddly shaped EDs carry crosswalk noise.
- **Electorates differ.** Primary: registered Democrats, ranked-choice. 2025 general: three-way, off-year, ~a third smaller than presidential. Turnout ratios are reported so the reader can see which neighborhoods dropped out.
- **ED boundary drift** across nyed_24d / 25b / 25d; matched on AD-ED number; EDs "combined into" another are dropped.
- **Demographics are 2020 Census population, not 2024–25 electorate**; block-group ACS income and education are 2019–2023 averages.